# Satellite Live Data Collector
## Telemetry + Commands from Ground Station APIs

Collects live data and stores as CSV for the Neural CDE pipeline.

**Outputs:** `telemetry.csv`, `commands.csv`

## 1. Configuration

In [1]:
import requests, json, time, os, csv, logging, re, shutil
from datetime import datetime, timedelta
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    'TELEMETRY_URL': 'http://172.20.10.1:9000/pid_info',
    'COMMANDS_URL': 'http://172.20.10.1:8888/PEP/summ.jsp',
    'SPACECRAFT_ID': 'EOS',
    'POLL_INTERVAL_SEC': 10,
    'COMMAND_FETCH_INTERVAL_SEC': 60,
    'OUTPUT_DIR': './live_data',
    'ARCHIVE_DIR': './live_data/archive',
    'TIMEOUT_SEC': 5,
    'MAX_RETRIES': 3,
    'RETRY_DELAY_SEC': 2,
    'PARAM_FILTER': [],
}

Path(CONFIG['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['ARCHIVE_DIR']).mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('collector')
print('Configuration loaded.')


Configuration loaded.


## 2. Telemetry Fetcher

In [2]:
class TelemetryClient:
    # Fetches live telemetry JSON from ground station API
    # Response: [{pid, mnemonic, units, upper_limit, lower_limit, tolerance, value}, ...]

    def __init__(self, config):
        self.url = config['TELEMETRY_URL']
        self.sc_id = config['SPACECRAFT_ID']
        self.timeout = config['TIMEOUT_SEC']
        self.retries = config['MAX_RETRIES']
        self.delay = config['RETRY_DELAY_SEC']
        self.param_filter = set(config.get('PARAM_FILTER', []))
        self.metadata = {}

    def fetch(self):
        for attempt in range(self.retries):
            try:
                resp = requests.get(f'{self.url}?sc_id={self.sc_id}', timeout=self.timeout)
                resp.raise_for_status()
                data = resp.json()
                if not isinstance(data, list): return []

                ts = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S')
                rows = []
                for item in data:
                    m = item.get('mnemonic', item.get('pid', ''))
                    if self.param_filter and m not in self.param_filter: continue
                    self.metadata[m] = {k: item.get(k) for k in ('pid','units','upper_limit','lower_limit','tolerance')}
                    val = item.get('value')
                    if val is None: continue
                    rows.append({'timestamp': ts, 'parameter_name': m, 'value': float(val)})
                return rows
            except requests.exceptions.ConnectionError:
                log.warning(f'Telemetry connection failed (attempt {attempt+1})')
                if attempt < self.retries - 1: time.sleep(self.delay)
            except Exception as e:
                log.error(f'Telemetry error: {e}')
                return []
        return []

telem_client = TelemetryClient(CONFIG)
print(f'Telemetry client: {CONFIG["TELEMETRY_URL"]}')


Telemetry client: http://172.20.10.1:9000/pid_info


## 3. Commands Fetcher (HTML Scraping)

In [3]:
class CommandClient:
    # Scrapes command history from PEP summary JSP page
    # Table columns: STEP.ID | INSTRUCTION | INFO (command) | TIME

    def __init__(self, config):
        self.url = config['COMMANDS_URL']
        self.sc = config['SPACECRAFT_ID']
        self.timeout = config['TIMEOUT_SEC']
        self.retries = config['MAX_RETRIES']
        self.delay = config['RETRY_DELAY_SEC']

    def fetch(self, start_time=None, end_time=None):
        if start_time is None:
            start_time = (datetime.utcnow() - timedelta(hours=1)).strftime('%Y-%m-%d+%H:%M:%S')
        if end_time is None:
            end_time = datetime.utcnow().strftime('%Y-%m-%d+%H:%M:%S')

        params = {'Sename': self.sc, 'STime': start_time, 'ETime': end_time}
        for attempt in range(self.retries):
            try:
                resp = requests.get(self.url, params=params, timeout=self.timeout)
                resp.raise_for_status()
                return self._parse(resp.text)
            except requests.exceptions.ConnectionError:
                log.warning(f'Commands connection failed (attempt {attempt+1})')
                if attempt < self.retries - 1: time.sleep(self.delay)
            except Exception as e:
                log.error(f'Command error: {e}')
                return []
        return []

    def _parse(self, html):
        soup = BeautifulSoup(html, 'html.parser')
        rows = []
        table = soup.find('table')
        if table is None:
            # Fallback: try JSON
            try:
                data = json.loads(html)
                return [{'timestamp': d.get('TIME',''), 'command_name': d.get('INFO','')} for d in data]
            except: return []

        for tr in table.find_all('tr')[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
            if len(cols) >= 4:
                ts = self._parse_ts(cols[3])
                if ts and cols[2]:
                    rows.append({'timestamp': ts, 'command_name': cols[2].strip()})
        log.info(f'Parsed {len(rows)} commands')
        return rows

    def _parse_ts(self, s):
        for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%Y-%m-%d+%H:%M:%S',
                     '%d-%m-%Y %H:%M:%S', '%Y/%m/%d %H:%M:%S', '%Y-%m-%d %H:%M:%S.%f']:
            try: return datetime.strptime(s.strip(), fmt).strftime('%Y-%m-%d %H:%M:%S')
            except ValueError: continue
        return None

cmd_client = CommandClient(CONFIG)
print(f'Command client: {CONFIG["COMMANDS_URL"]}')


Command client: http://172.20.10.1:8888/PEP/summ.jsp


## 4. CSV Writer (Daily Rotation)

In [4]:
class DataWriter:
    def __init__(self, out_dir, arc_dir):
        self.out = Path(out_dir); self.arc = Path(arc_dir)
        self.n_telem = 0; self.n_cmd = 0

    def write_telemetry(self, rows):
        if not rows: return 0
        today = datetime.utcnow().strftime('%Y%m%d')
        path = self.out / f'telemetry_{today}.csv'
        needs_hdr = not path.exists()
        with open(path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['timestamp','parameter_name','value'])
            if needs_hdr: w.writeheader()
            for r in rows: w.writerow({k: r[k] for k in ['timestamp','parameter_name','value']})
        self.n_telem += len(rows)
        shutil.copy2(path, self.out / 'telemetry.csv')
        return len(rows)

    def write_commands(self, rows):
        if not rows: return 0
        today = datetime.utcnow().strftime('%Y%m%d')
        path = self.out / f'commands_{today}.csv'
        existing = set()
        if path.exists():
            for r in csv.DictReader(open(path)):
                existing.add((r.get('timestamp',''), r.get('command_name','')))
        new = [r for r in rows if (r.get('timestamp',''), r.get('command_name','')) not in existing]
        if not new: return 0
        needs_hdr = not path.exists()
        with open(path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['timestamp','command_name'])
            if needs_hdr: w.writeheader()
            for r in new: w.writerow({k: r[k] for k in ['timestamp','command_name']})
        self.n_cmd += len(new)
        shutil.copy2(path, self.out / 'commands.csv')
        return len(new)

writer = DataWriter(CONFIG['OUTPUT_DIR'], CONFIG['ARCHIVE_DIR'])
print(f'Writer: {CONFIG["OUTPUT_DIR"]}')


Writer: ./live_data


## 5. Collection Loop

Run this cell to start. Stop with Ctrl+C.

In [ ]:
def collect_continuous(duration_hours=None):
    start = time.time()
    end = start + duration_hours * 3600 if duration_hours else float('inf')
    poll = CONFIG['POLL_INTERVAL_SEC']
    cmd_iv = CONFIG['COMMAND_FETCH_INTERVAL_SEC']
    last_cmd = 0; cycle = 0

    log.info(f'Starting collection (poll={poll}s)')
    try:
        while time.time() < end:
            cycle += 1; t0 = time.time()
            n_t = writer.write_telemetry(telem_client.fetch())
            n_c = 0
            if time.time() - last_cmd >= cmd_iv:
                n_c = writer.write_commands(cmd_client.fetch())
                last_cmd = time.time()
            if cycle % max(1, 30 // poll) == 0:
                elapsed = (time.time() - start) / 60
                log.info(f'[{elapsed:.1f}m] T:{writer.n_telem:,} C:{writer.n_cmd:,}')
            time.sleep(max(0, poll - (time.time() - t0)))
    except KeyboardInterrupt:
        log.info('Stopped by user')
    print(f'Done: {writer.n_telem:,} telemetry, {writer.n_cmd:,} commands')

# === Uncomment one: ===
collect_continuous(duration_hours=8)   # Run 8 hours
# collect_continuous()                   # Run forever
print('Uncomment a line above to start collection.')
print(f'Output: {CONFIG["OUTPUT_DIR"]}/telemetry.csv, commands.csv')


Uncomment a line above to start collection.
Output: ./live_data/telemetry.csv, commands.csv


## 6. Manual JSON-to-CSV Converter

For offline conversion of API responses:

In [6]:
def convert_telem_json(json_input, output='telemetry.csv', timestamp=None):
    if isinstance(json_input, str):
        if os.path.exists(json_input):
            with open(json_input) as f: json_input = json.load(f)
        else: json_input = json.loads(json_input)
    if timestamp is None: timestamp = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S')
    rows = [{'timestamp': timestamp, 'parameter_name': d.get('mnemonic', d.get('pid','')),
             'value': float(d.get('value', 0))} for d in json_input if d.get('value') is not None]
    pd.DataFrame(rows).to_csv(output, index=False)
    print(f'Saved {len(rows)} rows to {output}')
    return pd.DataFrame(rows)

# Example:
# convert_telem_json('api_response.json', 'telemetry.csv')
# convert_telem_json([{'mnemonic':'TEMP','value':25.0}], 'test.csv', '2026-03-26T08:00:00')
print('Manual converter ready. Call convert_telem_json() with your data.')


Manual converter ready. Call convert_telem_json() with your data.
